# Milestone 1: Data Profiling

Profil awal dataset besar Indonesia menggunakan Polars LazyFrame dan DuckDB. Isi sumber serta deskripsi dataset pada `data/README.md` sebelum menjalankan notebook ini.

In [1]:
from pathlib import Path

import duckdb
import polars as pl

PROJECT_ROOT = Path.cwd().resolve()
DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'click_stream.csv'

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'Dataset tidak ditemukan: {DATA_PATH}. '
        'Unduh data ke data/raw/ lalu sesuaikan DATA_PATH.'
    )

print(f'Dataset: {DATA_PATH.name}')
print(f'Ukuran file: {DATA_PATH.stat().st_size / 1_000_000:.1f} MB')

Dataset: click_stream.csv
Ukuran file: 1715.3 MB


## Profil Polars

`scan_parquet` hanya membangun rencana query; file baru dibaca ketika `collect()` dipanggil.

In [2]:
lazy_data = pl.scan_csv(DATA_PATH)
print(lazy_data.collect_schema())

ringkasan = lazy_data.select([
    pl.len().alias('jumlah_baris'),
    pl.all().null_count().name.suffix('_null'),
]).collect()
ringkasan

Schema({'session_id': String, 'event_name': String, 'event_time': String, 'event_id': String, 'traffic_source': String, 'event_metadata': String})


jumlah_baris,session_id_null,event_name_null,event_time_null,event_id_null,traffic_source_null,event_metadata_null
u32,u32,u32,u32,u32,u32,u32
12833602,0,0,0,0,0,8544062


In [3]:
contoh_data = lazy_data.head(10).collect()
contoh_data

session_id,event_name,event_time,event_id,traffic_source,event_metadata
str,str,str,str,str,str
"""fb0abf9e-fd1a-44dd-b5c0-2834d5…","""HOMEPAGE""","""2019-09-06T15:54:32.821085Z""","""9c4388c4-c95b-4678-b5ca-e9cbc0…","""MOBILE""",null
"""fb0abf9e-fd1a-44dd-b5c0-2834d5…","""SCROLL""","""2019-09-06T16:03:57.821085Z""","""4690e1f5-3f99-42d3-84a5-22c4c4…","""MOBILE""",null
"""7d440441-e67a-4d36-b324-80ffd6…","""HOMEPAGE""","""2019-09-01T12:05:10.322763Z""","""88aeaeb5-ec98-4859-852c-8abb48…","""MOBILE""",null
"""7d440441-e67a-4d36-b324-80ffd6…","""ADD_TO_CART""","""2019-09-01T12:06:33.322763Z""","""934e306e-ecc6-472f-9ccb-12c853…","""MOBILE""","""{'product_id': 15315, 'quantit…"
"""7d440441-e67a-4d36-b324-80ffd6…","""BOOKING""","""2019-09-01T12:15:29.425431Z""","""9f4767a1-40fa-4c9c-9524-dfad18…","""MOBILE""","""{'payment_status': 'Success'}"""
"""7d440441-e67a-4d36-b324-80ffd6…","""SEARCH""","""2019-09-01T12:07:00.322763Z""","""c952142b-4fe9-4694-ad7f-21a5d1…","""MOBILE""","""{'search_keywords': 'Dress Kon…"
"""7d440441-e67a-4d36-b324-80ffd6…","""HOMEPAGE""","""2019-09-01T12:08:27.322763Z""","""365b3840-9647-4bf5-917f-f0bec3…","""MOBILE""",null
"""7d440441-e67a-4d36-b324-80ffd6…","""ITEM_DETAIL""","""2019-09-01T12:09:49.322763Z""","""1a1e3548-108e-4520-bf04-1b01d4…","""MOBILE""",null
"""7d440441-e67a-4d36-b324-80ffd6…","""SCROLL""","""2019-09-01T12:11:06.322763Z""","""f7246095-b094-46c1-b2ac-bcfd08…","""MOBILE""",null


## Profil DuckDB

DuckDB membaca file Parquet langsung dari disk dan menghasilkan ringkasan statistik setiap kolom.

In [4]:
connection = duckdb.connect()
profil_duckdb = connection.sql(
    f"SUMMARIZE SELECT * FROM read_csv_auto('{DATA_PATH}')"
).pl()
profil_duckdb

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,null,str,str,str,i64,"decimal[9,2]"
"""session_id""","""VARCHAR""","""00001f29-3ede-4445-95f0-017893…","""fffffaf6-55ee-428b-a3bf-075cd7…",951924,null,null,null,null,null,12833602,0.00
"""event_name""","""VARCHAR""","""ADD_PROMO""","""SEARCH""",9,null,null,null,null,null,12833602,0.00
"""event_time""","""TIMESTAMP WITH TIME ZONE""","""2016-06-30 22:59:36.254358+00""","""2022-08-01 00:49:59.408424+00""",12842774,"""2020-10-17 06:11:58.513397+00""",null,"""2019-11-17 13:01:38.353914+00""","""2021-02-15 03:58:12.115666+00""","""2021-12-14 19:21:07.411068+00""",12833602,0.00
"""event_id""","""VARCHAR""","""0000001d-f3aa-498c-ad33-10e1fc…","""fffffc8c-b087-4025-8dfb-769e8b…",12716664,null,null,null,null,null,12833602,0.00
"""traffic_source""","""VARCHAR""","""MOBILE""","""WEB""",2,null,null,null,null,null,12833602,0.00
"""event_metadata""","""VARCHAR""","""{'payment_status': 'Failed'}""","""{'search_keywords': 'Topi'}""",2009445,null,null,null,null,null,12833602,66.58


## Catatan Milestone 1

Tulis temuan awal di bawah ini: ukuran data, jumlah baris, tipe kolom, kolom dengan null terbanyak, serta anomali yang perlu ditangani pada Milestone 2.

## Catatan Milestone 1

Berdasarkan hasil awal pada `data/raw/click_stream.csv` menggunakan Polars dan DuckDB:

* **Ukuran & volume data:** 1.715,3 MB (~1,72 GB) dengan total **12.833.602 baris** dan ada **6 kolom** (`session_id`, `event_name`, `event_time`, `event_id`, `traffic_source`, `event_metadata`).
* **Periode waktu:** `2016-06-30 22:59:36 UTC` hingga `2022-08-01 00:49:59 UTC`.
* **Tipe data:** Pada Polars (`scan_csv`), seluruh 6 kolom terbaca sebagai `String` (termasuk `event_time`), sedangkan DuckDB (`read_csv_auto`) otomatis mengenali `event_time` sebagai `TIMESTAMP WITH TIME ZONE`.
* **Missing values terbanyak:** terdapat pada kolom `event_metadata` yang memiliki nilai null yaitu sebanyak **66,58%**.
* **Anomali & rencana penanganan di Milestone 2:**
  1. Konversi tipe data `event_time` dari `String` ke `Datetime` (UTC/WIB) serta optimasi kolom kategorikal (`event_name`, `traffic_source`) ke `Categorical` agar hemat memori dan siap disimpan ke format Parquet.
  2. Parsing kolom `event_metadata` yang berformat string dictionary Python dengan tanda kutip tunggal misalnya `{'payment_status': 'Success'}`, bukan format JSON standar, untuk mengekstrak field penting seperti `product_id`, `quantity`, `payment_status`, atau `search_keywords`.
  3. Verifikasi duplikasi eksak pada `event_id` serta pengurutan ulang kronologis berdasarkan `session_id` dan `event_time`.
